<a href="https://colab.research.google.com/github/edwardoughton/spatial_computing/blob/main/09_01_Loading_and_Processing_OSM_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 9: Downloading data and answering spatial questions

In Week 8 we processed administrative boundaries. This week we download data (e.g., building footprints) so we can get started turning questions into answers, using evidence.

This will be important for your coursework project. 

**Learning goals:** learn how to inspect downloaded data, check geometry quality, project and clip features, calculate areas, select by size and distance, and support an answer with a table and map.


## 1. Understand the source

[OpenStreetMap (OSM)](https://www.openstreetmap.org) is a volunteer-built map using tags such as `building=yes` and `amenity=cafe`. Completeness varies by place and feature type. We can download and get started with this data. 

**This notebook downloads Overture Maps buildings, not data directly from an OSM API.** Overture combines multiple sources, including OSM. Do not describe every footprint as an OSM observation. See the [buildings guide](https://docs.overturemaps.org/guides/buildings/) and [Python API](https://github.com/OvertureMaps/overturemaps-py/blob/main/docs/python-api.md).

As a start, we can try download building footprints for a small area, e.g., GMU Fairfax campus. These represent building outlines, not total floor area, occupancy, population, or other conditions. Our study rectangle is near campus, and it is not an official GMU property boundary (it's small, so it's quick to download).


In [ ]:
# Run to install dependencies for the notebook.
%pip -q install "overturemaps[geopandas]" matplotlib

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from shapely.geometry import box, Point
from shapely.ops import unary_union
from IPython.display import display

# Longitude/latitude order: west, south, east, north.
bbox = (-77.315, 38.828, -77.300, 38.835)
work_dir = Path("week09_outputs")
work_dir.mkdir(exist_ok=True)
raw_path = work_dir / "gmu_buildings.gpkg"
metadata_path = work_dir / "download_metadata.json"
credit = "Building data: Overture Maps Foundation / © OpenStreetMap contributors"

## 2. Download once, save, and inspect

The download may take a few minutes. A bounding-box query can return footprints crossing the box edge; we will clip them later. Overture releases change, so we save a local extract and download metadata.

The source includes nested fields such as `sources`. Inspect these in the download. For straightforward GeoPackage export, our saved analysis extract retains **only ID and geometry**, not the complete source attributes.

To change the bounding box, use a **new output folder** so the cached extract is not mistaken for a new download.

In [ ]:
if raw_path.exists():
    print(f"Using saved extract: {raw_path}")
    if metadata_path.exists():
        print(metadata_path.read_text(encoding="utf-8"))
    else:
        print("Download date unknown: no metadata file found.")
else:
    from overturemaps import geodataframe
    import importlib.metadata

    downloaded = geodataframe("building", bbox=bbox)
    if downloaded is None or downloaded.empty:
        raise ValueError("No buildings returned. Check the bounding box and service.")
    if downloaded.crs is None:
        raise ValueError("Missing source CRS: verify it before assigning one.")
    display(downloaded.head(3))
    if "sources" in downloaded.columns:
        display(downloaded[["id", "sources"]].head(3))
    downloaded[["id", "geometry"]].to_file(raw_path, layer="buildings", driver="GPKG")
    metadata = {
        "source": "Overture Maps building theme",
        "accessed_utc": datetime.now(timezone.utc).isoformat(),
        "bbox_west_south_east_north": list(bbox),
        "client_version": importlib.metadata.version("overturemaps"),
        "release": "Client default at download time; not pinned",
        "saved_fields": ["id", "geometry"],
    }
    metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    print(f"Saved {len(downloaded)} downloaded records.")

In [ ]:
buildings = gpd.read_file(raw_path, layer="buildings")
if buildings.empty:
    raise ValueError("The saved extract is empty.")
if buildings.crs is None:
    raise ValueError("The saved data have no CRS. Verify the source before continuing.")
print("Rows and columns:", buildings.shape)
print("CRS:", buildings.crs)
print(buildings.geom_type.value_counts())
display(buildings.head())

## 3. Check quality, project, and clip

**Pause:** why would calculating `.area` directly in longitude and latitude be a problem?

Count missing, empty, invalid, and non-polygon geometries and repeated IDs. For this lesson we exclude unusable geometries rather than silently repair them; a real project should investigate the exclusions. We deduplicate usable records by ID, assuming an ID identifies one feature. Different IDs can still have overlapping geometry.

Use **EPSG:32618 (UTM zone 18N)** for meter-based measurements near Fairfax. `to_crs()` transforms coordinates. `set_crs()` assigns a label without transforming coordinates. Web Mercator is useful for web maps; this lesson uses a local projected CRS for measurement.

In [ ]:
quality = pd.Series({
    "downloaded_records": len(buildings),
    "missing_geometry": int(buildings.geometry.isna().sum()),
    "empty_geometry": int(buildings.geometry.is_empty.sum()),
    "invalid_nonmissing_geometry": int((buildings.geometry.notna() & ~buildings.geometry.is_valid).sum()),
    "non_polygon_nonmissing_geometry": int((buildings.geometry.notna() & ~buildings.geom_type.isin(["Polygon", "MultiPolygon"])).sum()),
    "repeated_ids": int(buildings["id"].duplicated().sum()),
})
display(quality.to_frame("count"))
# Diagnostic categories may overlap: do not add them to estimate exclusions.
usable = (buildings.geometry.notna() & ~buildings.geometry.is_empty
          & buildings.geometry.is_valid
          & buildings.geom_type.isin(["Polygon", "MultiPolygon"]))
clean = buildings.loc[usable].drop_duplicates(subset="id").copy()
if clean.empty:
    raise ValueError("No usable building polygons remain.")
print(f"Retained {len(clean)} of {len(buildings)} records after checks and ID deduplication.")
local_crs = "EPSG:32618"
full_buildings = clean.to_crs(local_crs)
full_buildings["full_area_m2"] = full_buildings.geometry.area
study_area = gpd.GeoDataFrame({"name": ["Campus-area study rectangle"]},
                             geometry=[box(*bbox)], crs="EPSG:4326").to_crs(local_crs)
study_polygon = study_area.geometry.iloc[0]
# Preserve full footprints and clip a separate copy for coverage calculations.
clipped = gpd.clip(full_buildings, study_area, keep_geom_type=True)
clipped = clipped.loc[clipped.geometry.area > 0].copy()
if clipped.empty:
    raise ValueError("No positive-area footprints intersect the study rectangle.")
clipped["area_in_study_m2"] = clipped.geometry.area
print(f"{len(clipped)} footprints have positive area inside the study rectangle.")

## 4. Worked question: how much of the study area is covered by buildings?

**Method:** divide the union area of clipped footprints by the study-area size and multiply by 100. A union counts overlapping footprints once; adding individual areas could double-count overlaps.

This measures **mapped footprint coverage**, not developed land or floor area. Roads and parking are not included. Do not label the uncovered share green space.

In [ ]:
study_area_m2 = study_polygon.area
covered_geometry = unary_union(clipped.geometry.tolist())
covered_area_m2 = covered_geometry.area
coverage_pct = 100 * covered_area_m2 / study_area_m2
summary = pd.DataFrame({
    "measure": ["Intersecting footprint records", "Study area (m²)",
                "Mapped building coverage (m²)", "Mapped building coverage (%)"],
    "value": [len(clipped), study_area_m2, covered_area_m2, coverage_pct],
})
display(summary.round(2))
print(f"Within the {study_area_m2:,.0f} m² study rectangle, "
      f"{len(clipped)} mapped footprints cover {covered_area_m2:,.0f} m² "
      f"({coverage_pct:.1f}%). Overlaps are counted once.")

### Which footprints contribute the most area inside the rectangle?

Rank by `area_in_study_m2`, rather than the full area of a building extending outside the boundary. Display both measures to make that distinction visible. A large footprint does not necessarily mean a tall building or a large population.

In [ ]:
largest = clipped.nlargest(5, "area_in_study_m2").copy()
display(largest[["id", "full_area_m2", "area_in_study_m2"]].round(1))
fig, ax = plt.subplots(figsize=(9, 7))
study_area.boundary.plot(ax=ax, color="black", linewidth=1.5)
clipped.plot(ax=ax, color="lightgray", edgecolor="white", linewidth=0.4)
largest.plot(ax=ax, color="darkorange", edgecolor="black", linewidth=0.5)
for rank, geom in enumerate(largest.geometry, start=1):
    p = geom.representative_point()
    ax.annotate(str(rank), (p.x, p.y), fontsize=9)
ax.set_title("Five largest mapped footprints inside the study rectangle")
ax.set_xlabel("Easting (m), UTM zone 18N")
ax.set_ylabel("Northing (m), UTM zone 18N")
fig.text(0.02, 0.01, credit, fontsize=8)
fig.tight_layout(rect=(0, 0.04, 1, 1))
fig.savefig(work_dir / "largest_footprints.png", dpi=180)
plt.show()

### Turn the results into an answer

The printed sentence gives the count, area, units, and percentage. The table and numbered map identify the largest footprints. Add the download date, study boundary definition, and a limitation to your written answer.

For example: ?These results describe mapped footprints within our study rectangle, not all GMU property. Missing or outdated footprints and the choice of boundary affect the result.? Do not infer population from footprint area alone.

## 5. Worked question: which buildings are within 100 meters of a meeting point?

Use the **hypothetical** point at longitude -77.3075, latitude 38.8315. This is a teaching location, not a verified entrance or assembly point.

**Selection rule:** the shortest straight-line distance from the point to the **full building footprint** is at most 100 meters. A point inside a building has distance zero. Candidates are restricted to buildings with positive footprint area inside the study rectangle; the search is not complete beyond that boundary.

Calculate distance to full footprints rather than clipped fragments. The buffer visualizes the rule. Footprints may touch the radius even when their centroids are farther away. This is proximity, not walking distance or travel time.

In [ ]:
meeting_point = gpd.GeoSeries([Point(-77.3075, 38.8315)], crs="EPSG:4326").to_crs(local_crs).iloc[0]
radius_m = 100
search_zone = gpd.GeoDataFrame(geometry=[meeting_point.buffer(radius_m)], crs=local_crs)
candidates = full_buildings.loc[full_buildings["id"].isin(clipped["id"])].copy()
candidates["distance_m"] = candidates.geometry.distance(meeting_point)
nearby = candidates.loc[candidates["distance_m"] <= radius_m].sort_values("distance_m").copy()
display(nearby[["id", "distance_m", "full_area_m2"]].round(1))
print(f"{len(nearby)} of {len(candidates)} candidate footprints are within {radius_m} m "
      "of the hypothetical point (straight-line distance to the footprint).")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
study_area.boundary.plot(ax=ax, color="black", linewidth=1)
candidates.plot(ax=ax, color="lightgray", edgecolor="white", linewidth=0.4)
search_zone.boundary.plot(ax=ax, color="royalblue", linewidth=2)
if not nearby.empty:
    nearby.plot(ax=ax, color="orange", edgecolor="black", linewidth=0.5)
gpd.GeoSeries([meeting_point], crs=local_crs).plot(ax=ax, color="red", marker="*", markersize=140)
ax.set_title(f"Footprints within {radius_m} m of a hypothetical meeting point")
ax.set_xlabel("Easting (m), UTM zone 18N")
ax.set_ylabel("Northing (m), UTM zone 18N")
fig.text(0.02, 0.01, credit, fontsize=8)
fig.tight_layout(rect=(0, 0.04, 1, 1))
fig.savefig(work_dir / "meeting_point_proximity.png", dpi=180)
plt.show()

## 6. Export evidence and check your results

Save the summary, ranked table, quality diagnostics, selected-building table, and processed footprints. Named outputs in `week09_outputs` are replaced when rerun. The processed GeoPackage contains scalar analysis fields, not the full source schema.

These checks catch impossible percentages and selections that violate our distance rule. They do not prove that the source includes every real building.

In [ ]:
assert clipped.crs.to_epsg() == 32618
assert (clipped["area_in_study_m2"] > 0).all()
assert 0 <= coverage_pct <= 100 + 1e-8
assert covered_area_m2 <= clipped["area_in_study_m2"].sum() + 1e-6
assert (nearby["distance_m"] <= radius_m).all()
summary.to_csv(work_dir / "coverage_summary.csv", index=False)
quality.to_csv(work_dir / "quality_checks.csv", header=["count"])
largest[["id", "full_area_m2", "area_in_study_m2"]].to_csv(work_dir / "largest_footprints.csv", index=False)
nearby[["id", "distance_m", "full_area_m2"]].to_csv(work_dir / "nearby_buildings.csv", index=False)
clipped.to_file(work_dir / "processed_buildings.gpkg", layer="footprints_in_study", driver="GPKG")
print("Checks passed. Outputs saved in:", work_dir.resolve())

## 7. Student processing tasks

For each task submit **code, a small result table, a map where useful, and 2-3 sentences answering the question**. State the units, selection rule, and one limitation. Exact counts depend on the saved download.

### Task A ? How sensitive is proximity to the radius?

Count candidate footprints within **50, 100, and 200 meters** of the same meeting point. Make a radius/count table and plot counts against radius. Explain why counts should never decrease as radius grows and why a larger search may reach beyond the study boundary.

*Hint:* loop over the radii and sum the Boolean condition on `candidates["distance_m"]`.

In [ ]:
# Your attempt: build a radius/count table. Keep the worked-example variables unchanged.

### Task B ? Which footprints are large enough for a closer look?

Select footprints with **at least 2,000 m² inside the study rectangle**. Report their count and the percentage of all intersecting footprint records they represent. Map them and compare their median full footprint area with that of the remaining candidates.

*Hint:* filter `clipped["area_in_study_m2"]`, then match IDs to `full_buildings` for full areas. If a group is empty, report that rather than interpret its missing median. Do not equate footprint size with occupancy.

In [ ]:
# Your attempt: select, summarize, and map large footprints.

### Task C ? Is coverage higher in the western or eastern half?

Split the rectangle at its longitude midpoint. Project both halves, clip buildings separately, and calculate **union-based coverage percentages**. Map the halves and say which has higher coverage.

*Hint:* create two `box()` polygons in EPSG:4326 and project them. Use each half's area as its denominator. A crossing building can contribute to both halves, so adding feature counts may double-count buildings.

In [ ]:
# Your attempt: construct the half-rectangles and compare coverage.

### Task D ? Does the selection rule change the answer?

Compare the 100-meter footprint-distance selection with one based on **building centroid distance**. List IDs selected by one rule but not the other and map the differences. Which rule identifies any footprint touching the radius?

*Hint:* use `candidates.geometry.centroid.distance(meeting_point)` in the projected CRS. A centroid need not be a building entrance.

In [ ]:
# Your attempt: compare footprint and centroid distance selections.

### Task E ? Answer your own question with another download

Choose another small bounding box and a **new output folder**. Before downloading, write a question, such as ?Which study rectangle has greater mapped footprint coverage?? Repeat the quality checks, projection, clipping, and coverage calculation. Compare percentages rather than raw totals if the study areas differ.

Optional extension: consult the [Overture schema](https://docs.overturemaps.org/schema/) and download another feature type, such as `place` or `segment`. Inspect the actual fields and geometry types first; building fields and polygon operations may not apply. Ask a suitable question, such as counting mapped places in a buffer. An Overture place query is not an OSM amenity query.

Explain how completeness, source dates, and study boundaries affect your comparison.

In [ ]:
# Your attempt: state a question, download separately, and support an answer with evidence.

## Review before submitting

- Does the written answer address the question with numbers and units?
- Do the table and map support the selection or comparison?
- Did you project before measuring and distinguish full from clipped footprints?
- Did you identify the source, download date, boundary, and limitations?

**Discussion:** What extra data would you need to estimate walking access, floor area, or people served? Why are footprints alone insufficient?